In [ ]:
from shapely.geometry import Polygon, mapping, shape,Point
import geopandas as gpd
import pandas as pd

### 서비스 인구 및 거주인구 비교
 - 거주인구와 비교하는 부분은 공주시 서비스 인구 데이터 사용 (2022년 12월 기준 인구로 사용)

In [ ]:
P1_Data_1 = 'C:/23project/A11/P1_Data/01.External/'
SKT_PATH = 'H:/23project/A11/P1_Data/01.External/공주시_생활인구데이터/SKT/'   #시점/파일명.csv
BC_PATH = 'H:/23project/A11/P1_Data/01.External/공주시_생활인구데이터/BC/'     #시점/파일명.csv
KCB_PATH = 'H:/23project/A11/P1_Data/01.External/공주시_생활인구데이터/KCB/'   #시점/파일명.csv
Tmap_PATH = 'H:/23project/A11/P1_Data/01.External/공주시_생활인구데이터/Tmap/' #파일명.csv

#### SKT 성연령별 유동인구 : GJ_SKT_SX_AGE_FLOW_NOPE
#### SKT 시간대별 유동인구 : GJ_SKT_TIME_FLOW_NOPE
#### SKT 요일별 유동인구 : GJ_SKT_DOW_FLOW_NOPE

In [ ]:
#연월별 서비스인구
pre_service_pop = pd.read_pickle('../P1_Data/02.Pretreatment/거주인구비교_서비스인구/pre_service_pop.pkl')
pre_service_hjd_pop = pd.read_pickle('../P1_Data/02.Pretreatment/거주인구비교_서비스인구/pre_service_hjd_pop.pkl')

In [ ]:
service_pop = pre_service_pop.copy()
service_hjd_pop = pre_service_hjd_pop.copy()

In [ ]:
#성연령별 유동인구(월별 격자합산)
pre_sx_age_pop = pd.read_pickle('../P1_Data/02.Pretreatment/성연령별_유동인구_월별/pre_sx_age_pop.pkl')
#시간대별 유동인구(월별 격자합산)
pre_time_pop = pd.read_pickle('../P1_Data/02.Pretreatment/시간대별_유동인구_월별/pre_time_pop.pkl')
#요일별 유동인구(월별 격자합산)
pre_day_pop = pd.read_pickle('../P1_Data/02.Pretreatment/요일별_유동인구_월별/pre_day_pop.pkl')

In [ ]:
ax_age_pop = pre_sx_age_pop.copy()
time_pop = pre_time_pop.copy()
day_pop = pre_day_pop.copy()

In [ ]:
# 'CRTR_YMD' 컬럼을 문자열로 변환
ax_age_pop['CRTR_YM'] = ax_age_pop['CRTR_YM'].astype(str)
# 'YEAR', 'MONTH', 'DAY' 컬럼 생성
ax_age_pop.insert(1, 'YEAR', ax_age_pop['CRTR_YM'].str[:4])
ax_age_pop.insert(2, 'MONTH', ax_age_pop['CRTR_YM'].str[4:6])

time_pop['CRTR_YM'] = time_pop['CRTR_YM'].astype(str)
# 'YEAR', 'MONTH', 'DAY' 컬럼 생성
time_pop.insert(1, 'YEAR', time_pop['CRTR_YM'].str[:4])
time_pop.insert(2, 'MONTH', time_pop['CRTR_YM'].str[4:6])

day_pop['CRTR_YM'] = day_pop['CRTR_YM'].astype(str)
# 'YEAR', 'MONTH', 'DAY' 컬럼 생성
day_pop.insert(1, 'YEAR', day_pop['CRTR_YM'].str[:4])
day_pop.insert(2, 'MONTH', day_pop['CRTR_YM'].str[4:6])

### 전년 동월대비 총 유동인구 비교

In [ ]:
service_pop_1 = service_pop.groupby('CRTR_YM')['total'].mean().round(1).reset_index()
#service_pop_1.to_csv('../P3_Result/01.CSV/1.공주시_생활인구패턴/전년_동월대비_총서비스인구비교(1006).csv',index=False)

### 성별 유동인구 패턴

In [ ]:
sx_pop_pattern1 = ax_age_pop.copy() #성연령별 유동인구

gender_age = ['ML_FLOW_PPLTN_CNT_10_BELOW', 'ML_FLOW_PPLTN_CNT_10_14',
       'ML_FLOW_PPLTN_CNT_15_19', 'ML_FLOW_PPLTN_CNT_20_24',
       'ML_FLOW_PPLTN_CNT_25_29', 'ML_FLOW_PPLTN_CNT_30_34',
       'ML_FLOW_PPLTN_CNT_35_39', 'ML_FLOW_PPLTN_CNT_40_44',
       'ML_FLOW_PPLTN_CNT_45_49', 'ML_FLOW_PPLTN_CNT_50_54',
       'ML_FLOW_PPLTN_CNT_55_59', 'ML_FLOW_PPLTN_CNT_60_64',
       'ML_FLOW_PPLTN_CNT_65_69', 'ML_FLOW_PPLTN_CNT_70_ABV',
       'FM_FLOW_PPLTN_CNT_10_BELOW', 'FM_FLOW_PPLTN_CNT_10_14',
       'FM_FLOW_PPLTN_CNT_15_19', 'FM_FLOW_PPLTN_CNT_20_24',
       'FM_FLOW_PPLTN_CNT_25_29', 'FM_FLOW_PPLTN_CNT_30_34',
       'FM_FLOW_PPLTN_CNT_35_39', 'FM_FLOW_PPLTN_CNT_40_44',
       'FM_FLOW_PPLTN_CNT_45_49', 'FM_FLOW_PPLTN_CNT_50_54',
       'FM_FLOW_PPLTN_CNT_55_59', 'FM_FLOW_PPLTN_CNT_60_64',
       'FM_FLOW_PPLTN_CNT_65_69', 'FM_FLOW_PPLTN_CNT_70_ABV']

# 성별 생활인구 컬럼 이름을 리스트로 저장합니다
female = ['FM_FLOW_PPLTN_CNT_10_BELOW', 'FM_FLOW_PPLTN_CNT_10_14',
       'FM_FLOW_PPLTN_CNT_15_19', 'FM_FLOW_PPLTN_CNT_20_24',
       'FM_FLOW_PPLTN_CNT_25_29', 'FM_FLOW_PPLTN_CNT_30_34',
       'FM_FLOW_PPLTN_CNT_35_39', 'FM_FLOW_PPLTN_CNT_40_44',
       'FM_FLOW_PPLTN_CNT_45_49', 'FM_FLOW_PPLTN_CNT_50_54',
       'FM_FLOW_PPLTN_CNT_55_59', 'FM_FLOW_PPLTN_CNT_60_64',
       'FM_FLOW_PPLTN_CNT_65_69', 'FM_FLOW_PPLTN_CNT_70_ABV']
male = ['ML_FLOW_PPLTN_CNT_10_BELOW', 'ML_FLOW_PPLTN_CNT_10_14',
       'ML_FLOW_PPLTN_CNT_15_19', 'ML_FLOW_PPLTN_CNT_20_24',
       'ML_FLOW_PPLTN_CNT_25_29', 'ML_FLOW_PPLTN_CNT_30_34',
       'ML_FLOW_PPLTN_CNT_35_39', 'ML_FLOW_PPLTN_CNT_40_44',
       'ML_FLOW_PPLTN_CNT_45_49', 'ML_FLOW_PPLTN_CNT_50_54',
       'ML_FLOW_PPLTN_CNT_55_59', 'ML_FLOW_PPLTN_CNT_60_64',
       'ML_FLOW_PPLTN_CNT_65_69', 'ML_FLOW_PPLTN_CNT_70_ABV']


# 성연령별 컬럼 값을 가로로 더해서 'Total' 컬럼에 추가합니다.
sx_pop_pattern1['성연령별_유동인구'] = sx_pop_pattern1[gender_age].sum(axis=1)
# 남자
sx_pop_pattern1['성연령별_유동인구_남'] = sx_pop_pattern1[male].sum(axis=1)
# 여자
sx_pop_pattern1['성연령별_유동인구_여'] = sx_pop_pattern1[female].sum(axis=1)

In [ ]:
#필요컬럼 추출
sx_pop_pattern1_1 = sx_pop_pattern1.copy()
sx_pop_pattern1_2 = sx_pop_pattern1_1.groupby(['CRTR_YM', 'YEAR', 'MONTH']).sum().round(1).reset_index()
sx_pop_pattern1_2 = sx_pop_pattern1_2[['CRTR_YM', 'YEAR', 'MONTH','성연령별_유동인구','성연령별_유동인구_남','성연령별_유동인구_여']]

sx_pop_pattern1_3 = sx_pop_pattern1_2.groupby(['YEAR']).mean().round(1).reset_index()
sx_pop_pattern1_3['유동인구_남_비율'] = (sx_pop_pattern1_3['성연령별_유동인구_남'] / sx_pop_pattern1_3['성연령별_유동인구'] * 100).round(1)
sx_pop_pattern1_3['유동인구_여_비율'] = (sx_pop_pattern1_3['성연령별_유동인구_여'] / sx_pop_pattern1_3['성연령별_유동인구'] * 100).round(1)
#sx_pop_pattern1_2.to_csv('../P3_Result/01.CSV/1.공주시_생활인구패턴/전체연도_월별_성연령별유동인구(1004).csv',index=False ,encoding='cp949')
#sx_pop_pattern1_3.to_csv('../P3_Result/01.CSV/1.공주시_생활인구패턴/전체연도_월평균_성연령별유동인구_성별비율(1004).csv',index=False ,encoding='cp949')

In [ ]:
# 연령대별 유동인구 패턴
sx_pop_pattern2 = ax_age_pop.copy() #성연령별 유동인구
sx_pop_pattern2 = sx_pop_pattern2[sx_pop_pattern2['CRTR_YM'] == '202212']

### 연령별 서비스인구 비교_(2022.12월 기준)

In [ ]:
#2022년 12월 기준 _ 5세별 주민등록인구 전처리
## 5세별 -> 10세별 수정작업
admin_pop = pd.read_csv(P1_Data_1+'kosis/행정구역_읍면동_성별_5세별_주민등록인구_2022년12월기준.csv',encoding='cp949',skiprows=1)
pre_admin_pop = admin_pop.copy()

pre_admin_pop['0 - 9세'] = pre_admin_pop['0 - 4세'] + pre_admin_pop['5 - 9세']
pre_admin_pop['10 - 19세'] = pre_admin_pop['10 - 14세'] + pre_admin_pop['15 - 19세']
pre_admin_pop['20 - 29세'] = pre_admin_pop['20 - 24세'] + pre_admin_pop['25 - 29세']
pre_admin_pop['30 - 39세'] = pre_admin_pop['30 - 34세'] + pre_admin_pop['35 - 39세']
pre_admin_pop['40 - 49세'] = pre_admin_pop['40 - 44세'] + pre_admin_pop['45 - 49세']
pre_admin_pop['50 - 59세'] = pre_admin_pop['50 - 54세'] + pre_admin_pop['55 - 59세']
pre_admin_pop['60세 이상'] = pre_admin_pop['60 - 64세'] + pre_admin_pop['65 - 69세'] + pre_admin_pop['70 - 74세'] + pre_admin_pop['75 - 79세'] + pre_admin_pop['80 - 84세'] + pre_admin_pop['85 - 89세'] + pre_admin_pop['90 - 94세'] + pre_admin_pop['95 - 99세'] + pre_admin_pop['100+']
pre_admin_pop = pre_admin_pop[['행정구역(동읍면)별','항목','계','0 - 9세','10 - 19세','20 - 29세','30 - 39세','40 - 49세','50 - 59세','60세 이상']]
pre_admin_pop.to_csv('../P3_Result/01.CSV/1.공주시_생활인구패턴/연령대별_주민등록인구(서비스인구비교)_202212기준(1006).csv',index=False,encoding='cp949')

In [ ]:
# 2022년 12월 기준 서비스인구
service_col = ['HOME_ML_PPLTN_CNT_10_BELOW', 'HOME_ML_PPLTN_CNT_10_14',
               'HOME_ML_PPLTN_CNT_15_19', 'HOME_ML_PPLTN_CNT_20_24',
               'HOME_ML_PPLTN_CNT_25_29', 'HOME_ML_PPLTN_CNT_30_34',
               'HOME_ML_PPLTN_CNT_35_39', 'HOME_ML_PPLTN_CNT_40_44',
               'HOME_ML_PPLTN_CNT_45_49', 'HOME_ML_PPLTN_CNT_50_54',
               'HOME_ML_PPLTN_CNT_55_59', 'HOME_ML_PPLTN_CNT_60_64',
               'HOME_ML_PPLTN_CNT_65_69', 'HOME_ML_PPLTN_CNT_70_ABV',
               'HOME_FM_PPLTN_CNT_10_BELOW', 'HOME_FM_PPLTN_CNT_10_14',
               'HOME_FM_PPLTN_CNT_15_19', 'HOME_FM_PPLTN_CNT_20_24',
               'HOME_FM_PPLTN_CNT_25_29', 'HOME_FM_PPLTN_CNT_30_34',
               'HOME_FM_PPLTN_CNT_35_39', 'HOME_FM_PPLTN_CNT_40_44',
               'HOME_FM_PPLTN_CNT_45_49', 'HOME_FM_PPLTN_CNT_50_54',
               'HOME_FM_PPLTN_CNT_55_59', 'HOME_FM_PPLTN_CNT_60_64',
               'HOME_FM_PPLTN_CNT_65_69', 'HOME_FM_PPLTN_CNT_70_ABV',
               'VST_ML_PPLTN_CNT_10_BELOW', 'VST_ML_PPLTN_CNT_10_14',
               'VST_ML_PPLTN_CNT_15_19', 'VST_ML_PPLTN_CNT_20_24',
               'VST_ML_PPLTN_CNT_25_29', 'VST_ML_PPLTN_CNT_30_34',
               'VST_ML_PPLTN_CNT_35_39', 'VST_ML_PPLTN_CNT_40_44',
               'VST_ML_PPLTN_CNT_45_49', 'VST_ML_PPLTN_CNT_50_54',
               'VST_ML_PPLTN_CNT_55_59', 'VST_ML_PPLTN_CNT_60_64',
               'VST_ML_PPLTN_CNT_65_69', 'VST_ML_PPLTN_CNT_70_ABV',
               'VST_FM_PPLTN_CNT_10_BELOW', 'VST_FM_PPLTN_CNT_10_14',
               'VST_FM_PPLTN_CNT_15_19', 'VST_FM_PPLTN_CNT_20_24',
               'VST_FM_PPLTN_CNT_25_29', 'VST_FM_PPLTN_CNT_30_34',
               'VST_FM_PPLTN_CNT_35_39', 'VST_FM_PPLTN_CNT_40_44',
               'VST_FM_PPLTN_CNT_45_49', 'VST_FM_PPLTN_CNT_50_54',
               'VST_FM_PPLTN_CNT_55_59', 'VST_FM_PPLTN_CNT_60_64',
               'VST_FM_PPLTN_CNT_65_69', 'VST_FM_PPLTN_CNT_70_ABV',
               'WRC_ML_PPLTN_CNT_10_BELOW', 'WRC_ML_PPLTN_CNT_10_14',
               'WRC_ML_PPLTN_CNT_15_19', 'WRC_ML_PPLTN_CNT_20_24',
               'WRC_ML_PPLTN_CNT_25_29', 'WRC_ML_PPLTN_CNT_30_34',
               'WRC_ML_PPLTN_CNT_35_39', 'WRC_ML_PPLTN_CNT_40_44',
               'WRC_ML_PPLTN_CNT_45_49', 'WRC_ML_PPLTN_CNT_50_54',
               'WRC_ML_PPLTN_CNT_55_59', 'WRC_ML_PPLTN_CNT_60_64',
               'WRC_ML_PPLTN_CNT_65_69', 'WRC_ML_PPLTN_CNT_70_ABV',
               'WRC_FM_PPLTN_CNT_10_BELOW', 'WRC_FM_PPLTN_CNT_10_14',
               'WRC_FM_PPLTN_CNT_15_19', 'WRC_FM_PPLTN_CNT_20_24',
               'WRC_FM_PPLTN_CNT_25_29', 'WRC_FM_PPLTN_CNT_30_34',
               'WRC_FM_PPLTN_CNT_35_39', 'WRC_FM_PPLTN_CNT_40_44',
               'WRC_FM_PPLTN_CNT_45_49', 'WRC_FM_PPLTN_CNT_50_54',
               'WRC_FM_PPLTN_CNT_55_59', 'WRC_FM_PPLTN_CNT_60_64',
               'WRC_FM_PPLTN_CNT_65_69', 'WRC_FM_PPLTN_CNT_70_ABV']
service_pop_202212 = pd.read_csv('H:/23project/A11/P1_Data/01.External/공주시_생활인구데이터/SKT/202212/GJ_SKT_SERVICE_SEX_AGE_PCELL_POP_202212.csv')
pre_sv_2212 = service_pop_202212.copy()
pre_sv_2212 = pre_sv_2212.groupby('CRTR_YMD')[service_col].sum().round(1).reset_index()

In [ ]:
pre_sv_2212['남0 - 9세s'] = pre_sv_2212['HOME_ML_PPLTN_CNT_10_BELOW'] + pre_sv_2212['VST_ML_PPLTN_CNT_10_BELOW'] + pre_sv_2212['WRC_ML_PPLTN_CNT_10_BELOW']
pre_sv_2212['남10 - 19세s'] = pre_sv_2212['HOME_ML_PPLTN_CNT_10_14'] + pre_sv_2212['HOME_ML_PPLTN_CNT_15_19'] + pre_sv_2212['VST_ML_PPLTN_CNT_10_14'] + pre_sv_2212['VST_ML_PPLTN_CNT_15_19'] + pre_sv_2212['WRC_ML_PPLTN_CNT_10_14'] + pre_sv_2212['VST_ML_PPLTN_CNT_15_19']
pre_sv_2212['남20 - 29세s'] = pre_sv_2212['HOME_ML_PPLTN_CNT_20_24'] + pre_sv_2212['HOME_ML_PPLTN_CNT_25_29'] + pre_sv_2212['VST_ML_PPLTN_CNT_20_24'] + pre_sv_2212['VST_ML_PPLTN_CNT_25_29'] + pre_sv_2212['WRC_ML_PPLTN_CNT_20_24'] + pre_sv_2212['VST_ML_PPLTN_CNT_25_29']
pre_sv_2212['남30 - 39세s'] = pre_sv_2212['HOME_ML_PPLTN_CNT_30_34'] + pre_sv_2212['HOME_ML_PPLTN_CNT_35_39'] + pre_sv_2212['VST_ML_PPLTN_CNT_30_34'] + pre_sv_2212['VST_ML_PPLTN_CNT_35_39'] + pre_sv_2212['WRC_ML_PPLTN_CNT_30_34'] + pre_sv_2212['VST_ML_PPLTN_CNT_35_39']
pre_sv_2212['남40 - 49세s'] = pre_sv_2212['HOME_ML_PPLTN_CNT_40_44'] + pre_sv_2212['HOME_ML_PPLTN_CNT_45_49'] + pre_sv_2212['VST_ML_PPLTN_CNT_40_44'] + pre_sv_2212['VST_ML_PPLTN_CNT_45_49'] + pre_sv_2212['WRC_ML_PPLTN_CNT_40_44'] + pre_sv_2212['VST_ML_PPLTN_CNT_45_49']
pre_sv_2212['남50 - 59세s'] = pre_sv_2212['HOME_ML_PPLTN_CNT_50_54'] + pre_sv_2212['HOME_ML_PPLTN_CNT_55_59'] + pre_sv_2212['VST_ML_PPLTN_CNT_50_54'] + pre_sv_2212['VST_ML_PPLTN_CNT_55_59'] + pre_sv_2212['WRC_ML_PPLTN_CNT_50_54'] + pre_sv_2212['VST_ML_PPLTN_CNT_55_59']
pre_sv_2212['남60세 이상s'] = pre_sv_2212['HOME_ML_PPLTN_CNT_60_64'] + pre_sv_2212['HOME_ML_PPLTN_CNT_65_69'] + pre_sv_2212['HOME_ML_PPLTN_CNT_70_ABV'] + pre_sv_2212['VST_ML_PPLTN_CNT_60_64'] + pre_sv_2212['VST_ML_PPLTN_CNT_65_69'] + pre_sv_2212['VST_ML_PPLTN_CNT_70_ABV'] + pre_sv_2212['WRC_ML_PPLTN_CNT_60_64'] + pre_sv_2212['WRC_ML_PPLTN_CNT_65_69'] + pre_sv_2212['WRC_ML_PPLTN_CNT_70_ABV']

pre_sv_2212['여0 - 9세s'] = pre_sv_2212['HOME_FM_PPLTN_CNT_10_BELOW'] + pre_sv_2212['VST_FM_PPLTN_CNT_10_BELOW'] + pre_sv_2212['WRC_FM_PPLTN_CNT_10_BELOW']
pre_sv_2212['여10 - 19세s'] = pre_sv_2212['HOME_FM_PPLTN_CNT_10_14'] + pre_sv_2212['HOME_FM_PPLTN_CNT_15_19'] + pre_sv_2212['VST_FM_PPLTN_CNT_10_14'] + pre_sv_2212['VST_FM_PPLTN_CNT_15_19'] + pre_sv_2212['WRC_FM_PPLTN_CNT_10_14'] + pre_sv_2212['VST_FM_PPLTN_CNT_15_19']
pre_sv_2212['여20 - 29세s'] = pre_sv_2212['HOME_FM_PPLTN_CNT_20_24'] + pre_sv_2212['HOME_FM_PPLTN_CNT_25_29'] + pre_sv_2212['VST_FM_PPLTN_CNT_20_24'] + pre_sv_2212['VST_FM_PPLTN_CNT_25_29'] + pre_sv_2212['WRC_FM_PPLTN_CNT_20_24'] + pre_sv_2212['VST_FM_PPLTN_CNT_25_29']
pre_sv_2212['여30 - 39세s'] = pre_sv_2212['HOME_FM_PPLTN_CNT_30_34'] + pre_sv_2212['HOME_FM_PPLTN_CNT_35_39'] + pre_sv_2212['VST_FM_PPLTN_CNT_30_34'] + pre_sv_2212['VST_FM_PPLTN_CNT_35_39'] + pre_sv_2212['WRC_FM_PPLTN_CNT_30_34'] + pre_sv_2212['VST_FM_PPLTN_CNT_35_39']
pre_sv_2212['여40 - 49세s'] = pre_sv_2212['HOME_FM_PPLTN_CNT_40_44'] + pre_sv_2212['HOME_FM_PPLTN_CNT_45_49'] + pre_sv_2212['VST_FM_PPLTN_CNT_40_44'] + pre_sv_2212['VST_FM_PPLTN_CNT_45_49'] + pre_sv_2212['WRC_FM_PPLTN_CNT_40_44'] + pre_sv_2212['VST_FM_PPLTN_CNT_45_49']
pre_sv_2212['여50 - 59세s'] = pre_sv_2212['HOME_FM_PPLTN_CNT_50_54'] + pre_sv_2212['HOME_FM_PPLTN_CNT_55_59'] + pre_sv_2212['VST_FM_PPLTN_CNT_50_54'] + pre_sv_2212['VST_FM_PPLTN_CNT_55_59'] + pre_sv_2212['WRC_FM_PPLTN_CNT_50_54'] + pre_sv_2212['VST_FM_PPLTN_CNT_55_59']
pre_sv_2212['여60세 이상s'] = pre_sv_2212['HOME_FM_PPLTN_CNT_60_64'] + pre_sv_2212['HOME_FM_PPLTN_CNT_65_69'] + pre_sv_2212['HOME_FM_PPLTN_CNT_70_ABV'] + pre_sv_2212['VST_FM_PPLTN_CNT_60_64'] + pre_sv_2212['VST_FM_PPLTN_CNT_65_69'] + pre_sv_2212['VST_FM_PPLTN_CNT_70_ABV'] + pre_sv_2212['WRC_FM_PPLTN_CNT_60_64'] + pre_sv_2212['WRC_FM_PPLTN_CNT_65_69'] + pre_sv_2212['WRC_FM_PPLTN_CNT_70_ABV']

pre2_sv_2212 = pre_sv_2212[['CRTR_YMD','남0 - 9세s','남10 - 19세s','남20 - 29세s','남30 - 39세s','남40 - 49세s','남50 - 59세s','남60세 이상s','여0 - 9세s','여10 - 19세s','여20 - 29세s','여30 - 39세s','여40 - 49세s','여50 - 59세s','여60세 이상s']]
pre2_sv_2212['S0-9세'] = pre2_sv_2212['남0 - 9세s'] + pre2_sv_2212['여0 - 9세s']
pre2_sv_2212['S10-19세'] = pre2_sv_2212['남10 - 19세s'] + pre2_sv_2212['여10 - 19세s']
pre2_sv_2212['S20-29세'] = pre2_sv_2212['남20 - 29세s'] + pre2_sv_2212['여20 - 29세s']
pre2_sv_2212['S30-39세'] = pre2_sv_2212['남30 - 39세s'] + pre2_sv_2212['여30 - 39세s']
pre2_sv_2212['S40-49세'] = pre2_sv_2212['남40 - 49세s'] + pre2_sv_2212['여40 - 49세s']
pre2_sv_2212['S50-59세'] = pre2_sv_2212['남50 - 59세s'] + pre2_sv_2212['여50 - 59세s']
pre2_sv_2212['S60세 이상'] = pre2_sv_2212['남60세 이상s'] + pre2_sv_2212['여60세 이상s']

#pre2_sv_2212 = pre2_sv_2212[['CRTR_YMD','S0-9세', 'S10-19세', 'S20-29세', 'S30-39세', 'S40-49세', 'S50-59세','S60세 이상']]
#pre2_sv_2212.to_csv('../P3_Result/01.CSV/1.공주시_생활인구패턴/연령대별_서비스인구(주민등록인구비교)_202212기준(1006).csv',index=False,encoding='cp949')

pre3_sv_2212 = pre2_sv_2212.transpose()
pre3_sv_2212.columns = pre3_sv_2212.iloc[0]
# 첫 번째 행 삭제
pre3_sv_2212 = pre3_sv_2212[1:]
#pre3_sv_2212.to_csv('../P3_Result/01.CSV/1.공주시_생활인구패턴/연령대별_서비스인구(주민등록인구비교)_202212기준(1010).csv',encoding='cp949')

In [ ]:
pre3_sv_2212

### 요일별 유동인구 패턴(연도별)

In [ ]:
day_pop1 = day_pop.copy()
day = ['Monday', 'Tuesday','Wednesday', 'Thursday', 'Friday', 'Saturday','Sunday']
day_pop1.columns = ['CRTR_YM','YEAR','MONTH','COORD_X','COORD_Y','Monday', 'Tuesday','Wednesday', 'Thursday', 'Friday', 'Saturday','Sunday']
day_pop1['유동인구'] = day_pop1[day].sum(axis=1)
# 연도별로 그룹화하고 각 요일별 유동인구 평균 계산
pre_day_pop1 = day_pop1.groupby(['YEAR','MONTH'])['Monday', 'Tuesday','Wednesday', 'Thursday', 'Friday', 'Saturday','Sunday','유동인구'].sum().round(1).reset_index()
pre_day_pop1.to_csv('../P3_Result/01.CSV/1.공주시_생활인구패턴/전체연도_요일별_월평균_유동인구(1010).csv',index = False,encoding='cp949')

### 시간대별 유동인구 패턴

In [ ]:
time_pop1 = time_pop.copy()
time = ['0', '1', '2', '3', '4', '5', '6', '7', '8',
       '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20',
       '21', '22', '23']
time_pop1.columns = ['CRTR_YM','YEAR','MONTH','COORD_X','COORD_Y','0', '1', '2', '3', '4', '5', '6', '7', '8','9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20','21', '22', '23']
time_pop1['유동인구'] = time_pop1[time].sum(axis=1)
pre_time_pop1 = time_pop1.groupby(['YEAR','MONTH'])[['0', '1', '2', '3', '4', '5', '6', '7', '8','9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20','21', '22', '23','유동인구']].sum().round(1).reset_index()
pre_time_pop1.to_csv('../P3_Result/01.CSV/1.공주시_생활인구패턴/전체연도_시간대별_월평균_유동인구(1010).csv',index = False,encoding='cp949')